# Notebook to fix PR issues


### Generate the UE data from mobility model first

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import pandas as pd
import scipy
import numpy as np
from radp_library import *
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from radp.digital_twin.mobility.param_regression import get_predicted_alpha,preprocess_ue_data
from radp.digital_twin.utils.cell_selection import perform_attachment
from radp.digital_twin.rf.bayesian.bayesian_engine import (
    BayesianDigitalTwin,
    NormMethod,
)
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
training_data = get_ue_data(params)
training_data.head()

In [ ]:
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90


topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180


topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

In [ ]:
def _prepare_all_UEs_from_all_cells_df(
     data, topology
    ) -> pd.DataFrame:
        """
        Connects each user equipment (UE) entry to all cells in the topology for each tick,
        effectively creating a Cartesian product of UEs and cells, which includes data from both sources.
        """

        ue_data = data
        ue_data = ue_data.rename(columns={"lat": "latitude", "lon": "longitude"})
        topology_tmp = topology
        
        """" 
        As 'ue_data' and 'topology_tmp' both have columns 'cell_id', we encounter a name conflict when merging 
        with column 'Key' (merge key is not set to cell_id). In order to resolve this, pandas adds a suffix to 
        the conflicting columns as 'cell_id_x' and 'cell_id_y'  
        """  
        if 'cell_id' in ue_data.columns:
            ue_data = ue_data.drop(columns = ['cell_id'])
        
        # Remove the 'cell_' prefix and convert cell_id to integer if needed
        if topology_tmp["cell_id"].dtype == object:
            topology_tmp["cell_id"] = (
                topology_tmp["cell_id"].str.replace("cell_", "").astype(int)
            )
        
        ue_data["key"] = 1
        topology_tmp["key"] = 1
        combined_df = pd.merge(ue_data, topology_tmp, on="key").drop("key", axis=1)
        print("Combined DataFrame:", combined_df)
        return combined_df


In [ ]:
def _preprocess_ue_topology_data(data,topology) -> pd.DataFrame:
        full_data = _prepare_all_UEs_from_all_cells_df(data,topology)
        full_data["log_distance"] = full_data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        full_data["cell_rxpwr_dbm"] = full_data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        return full_data

In [ ]:
def _preprocess_ue_training_data(data,topology) -> pd.DataFrame:
        data = _preprocess_ue_topology_data(data,topology)
        train_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
        desired_idxs = [1 + r for r in range(n_cell)]

        n_samples_train = []
        for df in train_per_cell_df:
            n_samples_train.append(df.shape[0])

        train_per_cell_df_processed = []
        for i in range(n_cell):
            train_per_cell_df_processed.append(
                get_percell_data(
                    data_in=train_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_train[i],
                )[0][0]
            )

        training_data = {}

        for i, df in enumerate(train_per_cell_df_processed):
            train_cell_id = idx_cell_id_mapping[i + 1]
            training_data[train_cell_id] = df

        for train_cell_id, training_data_idx in training_data.items():
            training_data_idx["cell_id"] = train_cell_id
            training_data_idx["cell_lat"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lat"].values[0]
            training_data_idx["cell_lon"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lon"].values[0]
            training_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_az_deg"].values[0]
            training_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            training_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    training_data_idx["cell_az_deg"].values[0],
                    training_data_idx["cell_lat"].values[0],
                    training_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    training_data_idx["latitude"], training_data_idx["longitude"]
                )
            ]

        return training_data

In [ ]:
bayesian_digital_twins = {}

In [ ]:
def _training(bayesian_digital_twins, maxiter: int, train_data: pd.DataFrame,topology: pd.DataFrame) -> List[float]:
        """
        Trains the Bayesian Digital Twins for each cell in the topology using the UE locations and features
        like log distance, relative bearing, and cell received power (Rx power).
        """
        training_data = _preprocess_ue_training_data(train_data,topology)
        loss_vs_iters = []
        for train_cell_id, training_data_idx in training_data.items():
            bayesian_digital_twins[train_cell_id] = BayesianDigitalTwin(
                data_in=[training_data_idx],
                x_columns=["log_distance", "relative_bearing"],
                y_columns=["cell_rxpwr_dbm"],
                norm_method=NormMethod.MINMAX,
            )
            bayesian_digital_twins[train_cell_id] = bayesian_digital_twins[
                train_cell_id
            ]
            loss_vs_iters.append(
                bayesian_digital_twins[train_cell_id].train_distributed_gpmodel(
                    maxiter=maxiter,
                )
            )
        return bayesian_digital_twins, loss_vs_iters

In [ ]:
bayesian_digital_twins, loss_vs_iters = _training(
    bayesian_digital_twins,
    maxiter=100,
    train_data=training_data,
    topology=topology,
)


In [ ]:
params2 = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 7,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 10,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.8,
                    "variance": 0.5,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
def _preprocess_prediction_data(pred_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(pred_data,topology)

        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )
        data["cell_rxpwr_dbm"] = data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        data["relative_bearing"] = data.apply(
            lambda row: GISTools.get_relative_bearing(
                row["cell_az_deg"],
                row["cell_lat"],
                row["cell_lon"],
                row["latitude"],
                row["longitude"],
            ),
            axis=1,
        )
        return data

In [ ]:
# Prediction
def _predictions(pred_data,topology,bayesian_digital_twins) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Predicts the received power for each User Equipment (UE) at different locations and ticks using Bayesian Digital Twins.
        It then determines the best cell for each UE to attach based on the predicted power values.
        """
        prediction_data = _preprocess_prediction_data(pred_data,topology)
        full_prediction_df = pd.DataFrame()

        # Loop over each 'tick'
        for tick, tick_df in prediction_data.groupby("tick"):
            # Loop over each 'cell_id' within the current 'tick'
            for cell_id, cell_df in tick_df.groupby("cell_id"):
                # Check if the Bayesian model for this cell_id exists
                if cell_id in bayesian_digital_twins:
                    # Perform the Bayesian prediction
                    pred_means_percell, _ = bayesian_digital_twins[
                        cell_id
                    ].predict_distributed_gpmodel(prediction_dfs=[cell_df])

                    # Assuming 'pred_means_percell' returns a list of predictions corresponding to the DataFrame index
                    cell_df["pred_means"] = pred_means_percell[0]

                    # Include additional necessary columns for the final DataFrame
                    cell_df["tick"] = tick
                    cell_df["cell_id"] = cell_id

                    # Append the predictions to the full DataFrame
                    full_prediction_df = pd.concat(
                        [full_prediction_df, cell_df], ignore_index=True
                    )
                else:
                    # Handle missing models, e.g., log a warning or initialize a default model
                    print(
                        f"No model available for cell_id {cell_id}, skipping prediction."
                    )

        full_prediction_df = full_prediction_df.rename(
            columns={"latitude": "loc_y", "longitude": "loc_x"}
        )
        predicted = perform_attachment(full_prediction_df, topology)

        return predicted, full_prediction_df

In [ ]:
prediction_data = get_ue_data(params2)

In [ ]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bayesian_digital_twins
)

### Disect all the codes from MRO and put it to radp_library.py


In [ ]:
bdt = {}

In [ ]:
update_data = pd.read_csv('data/sim_data/combined_data_dump.csv')
update_data.head()

## Attaching real **rxpower_dbm** data

In [ ]:
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90


topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180


topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

In [ ]:
real_rxdmb_data = pd.read_csv('data/sim_data/cell_rxpwr_debug.csv')
real_rxdmb_data.head()

In [ ]:
real_rxdmb_data.shape

In [ ]:
real_rxdmb_data[['longitude', 'latitude']].nunique()

In [ ]:
real_rxdmb_data[(real_rxdmb_data['longitude'] == 5.606533) & (real_rxdmb_data['latitude'] == 12.413489)]

In [ ]:
update_data = real_rxdmb_data.copy()
update_data.drop(columns=['mock_ue_id', 'tick', 'cell_lat', 'cell_lon', 'cell_az_deg', 'cell_carrier_freq_mhz', 'log_distance', 'relative_bearing'], inplace=True)
# update_data = update_data[['']]
update_data.head()

In [ ]:
def _preprocess_ue_update_data(update_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(update_data,topology)
        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        update_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))

        n_samples_update = []
        for df in update_per_cell_df:
            n_samples_update.append(df.shape[0])

        update_per_cell_df_processed = []
        for i in range(n_cell):
            update_per_cell_df_processed.append(
                get_percell_data(
                    data_in=update_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_update[i],
                )[0][0]
            )

        update_data = {}

        for i, df in enumerate(update_per_cell_df_processed):
            update_cell_id = idx_cell_id_mapping[i + 1]
            update_data[update_cell_id] = df

        for update_cell_id, update_data_idx in update_data.items():
            update_data_idx["cell_id"] = update_cell_id
            update_data_idx["cell_lat"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lat"].values[0]
            update_data_idx["cell_lon"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lon"].values[0]
            update_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_az_deg"].values[0]
            update_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            update_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    update_data_idx["cell_az_deg"].values[0],
                    update_data_idx["cell_lat"].values[0],
                    update_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    update_data_idx["latitude"], update_data_idx["longitude"]
                )
            ]
        return update_data

In [ ]:
def train_or_update_rf_twin(new_data: pd.DataFrame,topology: pd.DataFrame, bayesian_digital_twins):
        try:
            if not isinstance(new_data, pd.DataFrame):
                raise TypeError("The input 'new_data' must be a pandas DataFrame.")
            
            new_data = new_data.rename(
                columns={"cell_rxpower_dbm": "cell_rxpwr_dbm"}
            )
            expected_columns = {"longitude", "latitude", "cell_rxpwr_dbm"}
            if not expected_columns.issubset(new_data.columns):
                raise ValueError(
                    f"The input DataFrame must contain the following columns: {expected_columns}"
                )

            if bayesian_digital_twins:
                update_data = new_data
                updated_data = _preprocess_ue_update_data(update_data, topology)
                updated_data_list = list(updated_data.values())
                print("Updated_List ",updated_data_list)

                for data_idx, update_data_df in enumerate(updated_data_list):
                    update_cell_id = data_idx + 1
                    if update_cell_id in bayesian_digital_twins:
                        print(f"Updating cell {update_cell_id} with {len(update_data_df)} samples.")
                        bayesian_digital_twins[
                            update_cell_id
                        ].update_trained_gpmodel([update_data_df])
            else:
                print(
                    "No Bayesian Digital Twins available for update. Training from scratch."
                )
                new_data = new_data.drop(
                    columns=["cell_rxpower_dbm"], errors="ignore"
                )
                _training(bayesian_digital_twins,maxiter=100, train_data=new_data,topology = topology)
        
            # return bayesian_digital_twins
        
        except TypeError as te:
            print(f"TypeError: {te}")
        except ValueError as ve:
            print(f"ValueError: {ve}")
        except KeyError as ke:
            print(f"KeyError: {ke}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

In [ ]:
bdt

In [ ]:
updates = train_or_update_rf_twin(update_data,topology,bdt)

In [ ]:
bdt

In [ ]:
params2 = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 7,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 10,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.8,
                    "variance": 0.5,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
prediction_data = get_ue_data(params2)

In [ ]:
prediction_data.head()

In [ ]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bdt
)

In [ ]:
predictions.head()

In [ ]:
full_prediction_df.head()

# cell_rxpwr_dbm --> calculated
# rxpower_dbm --> predicted

In [ ]:
# Training from scratch done now will update the exisitng BDT
update_not_from_scratch = train_or_update_rf_twin(update_data,topology,bdt)

# Fresh Fixing

**"[Optionally] User provides Rx power data with UE lat/lon for training twins"**

Note:

1. consider all preprocessing of ue would be called from outside MRO (may be implement this later after the whole flow is implemented, for now call the preprocessing function as it is and define new functions where necessary)
2. `train_or_update_rf_twin()` have input `new_data` df and `bdt` (empty or populated)
    - `new_data` is ue data that client inputs to train/update `bdt`

cols of `new_data`:

- usually would be with [lat, lon]
    - the existing flow stays as it is like using the fspl function to calculate mock rx_power. just need to isolate the preprocessing part outside the 
- to achieve "[Optionally] User provides Rx power data with UE lat/lon for training twins": it should contain [lat, lon, cell_id, cell_rxpower_dbm] in a combined/cartesian format
    - when cell_rxpower_dbm is available, all the process of calculating mock rx_power using fspl function should be by passed for both case scenario of (train from scratch or update) in `train_or_update_rf_twin()`

**new understanding**:

- MRO requires combined format of data like [lat, lon, cell_id, cell_rxpwr_dbm], in ue x cell cartesian format
    - if client has `new_data` like [lat, lon] --> direct him to radp library/preprocess utility file from where he can get the desired format data.
    - else, client has `new_data` with [lat, lon, cell_id, cell_rxpwr_dbm], in ue x cell cartesian format already
    - this `new_data` is ready to pass in MRO

- MRO constructor accepts topology, mobility_model/mobility_params, bdt[optional]
- MRO has `train_or_update_rf_twin`, `solve` and `save` methods and `_training`, `_prediction` helper methods


## import & load functions

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import pandas as pd
import scipy
import numpy as np
from radp_library import *
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from radp.digital_twin.mobility.param_regression import get_predicted_alpha,preprocess_ue_data
from radp.digital_twin.utils.cell_selection import perform_attachment
from radp.digital_twin.rf.bayesian.bayesian_engine import (
    BayesianDigitalTwin,
    NormMethod,
)
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

In [ ]:
def _prepare_all_UEs_from_all_cells_df(
     data, topology
    ) -> pd.DataFrame:
        """
        Connects each user equipment (UE) entry to all cells in the topology for each tick,
        effectively creating a Cartesian product of UEs and cells, which includes data from both sources.
        """

        ue_data = data
        ue_data = ue_data.rename(columns={"lat": "latitude", "lon": "longitude"})
        topology_tmp = topology
        
        """" 
        As 'ue_data' and 'topology_tmp' both have columns 'cell_id', we encounter a name conflict when merging 
        with column 'Key' (merge key is not set to cell_id). In order to resolve this, pandas adds a suffix to 
        the conflicting columns as 'cell_id_x' and 'cell_id_y'  
        """  
        if 'cell_id' in ue_data.columns:
            ue_data = ue_data.drop(columns = ['cell_id'])
        
        # Remove the 'cell_' prefix and convert cell_id to integer if needed
        if topology_tmp["cell_id"].dtype == object:
            topology_tmp["cell_id"] = (
                topology_tmp["cell_id"].str.replace("cell_", "").astype(int)
            )
        
        ue_data["key"] = 1
        topology_tmp["key"] = 1
        combined_df = pd.merge(ue_data, topology_tmp, on="key").drop("key", axis=1)
        print("Combined DataFrame:", combined_df)
        return combined_df


In [ ]:
def _preprocess_ue_topology_data(data,topology) -> pd.DataFrame:
        full_data = _prepare_all_UEs_from_all_cells_df(data,topology)
        full_data["log_distance"] = full_data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        full_data["cell_rxpwr_dbm"] = full_data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        return full_data

In [ ]:
def _preprocess_ue_training_data(data,topology) -> pd.DataFrame:
        data = _preprocess_ue_topology_data(data,topology)
        train_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
        desired_idxs = [1 + r for r in range(n_cell)]

        n_samples_train = []
        for df in train_per_cell_df:
            n_samples_train.append(df.shape[0])

        train_per_cell_df_processed = []
        for i in range(n_cell):
            train_per_cell_df_processed.append(
                get_percell_data(
                    data_in=train_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_train[i],
                )[0][0]
            )

        training_data = {}

        for i, df in enumerate(train_per_cell_df_processed):
            train_cell_id = idx_cell_id_mapping[i + 1]
            training_data[train_cell_id] = df

        for train_cell_id, training_data_idx in training_data.items():
            training_data_idx["cell_id"] = train_cell_id
            training_data_idx["cell_lat"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lat"].values[0]
            training_data_idx["cell_lon"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lon"].values[0]
            training_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_az_deg"].values[0]
            training_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            training_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    training_data_idx["cell_az_deg"].values[0],
                    training_data_idx["cell_lat"].values[0],
                    training_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    training_data_idx["latitude"], training_data_idx["longitude"]
                )
            ]

        return training_data

In [ ]:
def _training(bayesian_digital_twins, maxiter: int, train_data: pd.DataFrame,topology: pd.DataFrame) -> List[float]:
        """
        Trains the Bayesian Digital Twins for each cell in the topology using the UE locations and features
        like log distance, relative bearing, and cell received power (Rx power).
        """
        training_data = _preprocess_ue_training_data(train_data,topology)
        loss_vs_iters = []
        for train_cell_id, training_data_idx in training_data.items():
            bayesian_digital_twins[train_cell_id] = BayesianDigitalTwin(
                data_in=[training_data_idx],
                x_columns=["log_distance", "relative_bearing"],
                y_columns=["cell_rxpwr_dbm"],
                norm_method=NormMethod.MINMAX,
            )
            bayesian_digital_twins[train_cell_id] = bayesian_digital_twins[
                train_cell_id
            ]
            loss_vs_iters.append(
                bayesian_digital_twins[train_cell_id].train_distributed_gpmodel(
                    maxiter=maxiter,
                )
            )
        return bayesian_digital_twins, loss_vs_iters

In [ ]:
def _preprocess_prediction_data(pred_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(pred_data,topology)

        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )
        data["cell_rxpwr_dbm"] = data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        data["relative_bearing"] = data.apply(
            lambda row: GISTools.get_relative_bearing(
                row["cell_az_deg"],
                row["cell_lat"],
                row["cell_lon"],
                row["latitude"],
                row["longitude"],
            ),
            axis=1,
        )
        return data

In [ ]:
# Prediction
def _predictions(pred_data,topology,bayesian_digital_twins) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Predicts the received power for each User Equipment (UE) at different locations and ticks using Bayesian Digital Twins.
        It then determines the best cell for each UE to attach based on the predicted power values.
        """
        prediction_data = _preprocess_prediction_data(pred_data,topology)
        full_prediction_df = pd.DataFrame()

        # Loop over each 'tick'
        for tick, tick_df in prediction_data.groupby("tick"):
            # Loop over each 'cell_id' within the current 'tick'
            for cell_id, cell_df in tick_df.groupby("cell_id"):
                # Check if the Bayesian model for this cell_id exists
                if cell_id in bayesian_digital_twins:
                    # Perform the Bayesian prediction
                    pred_means_percell, _ = bayesian_digital_twins[
                        cell_id
                    ].predict_distributed_gpmodel(prediction_dfs=[cell_df])

                    # Assuming 'pred_means_percell' returns a list of predictions corresponding to the DataFrame index
                    cell_df["pred_means"] = pred_means_percell[0]

                    # Include additional necessary columns for the final DataFrame
                    cell_df["tick"] = tick
                    cell_df["cell_id"] = cell_id

                    # Append the predictions to the full DataFrame
                    full_prediction_df = pd.concat(
                        [full_prediction_df, cell_df], ignore_index=True
                    )
                else:
                    # Handle missing models, e.g., log a warning or initialize a default model
                    print(
                        f"No model available for cell_id {cell_id}, skipping prediction."
                    )

        full_prediction_df = full_prediction_df.rename(
            columns={"latitude": "loc_y", "longitude": "loc_x"}
        )
        predicted = perform_attachment(full_prediction_df, topology)

        return predicted, full_prediction_df

In [ ]:
def _preprocess_ue_update_data(update_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(update_data,topology)
        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        update_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))

        n_samples_update = []
        for df in update_per_cell_df:
            n_samples_update.append(df.shape[0])

        update_per_cell_df_processed = []
        for i in range(n_cell):
            update_per_cell_df_processed.append(
                get_percell_data(
                    data_in=update_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_update[i],
                )[0][0]
            )

        update_data = {}

        for i, df in enumerate(update_per_cell_df_processed):
            update_cell_id = idx_cell_id_mapping[i + 1]
            update_data[update_cell_id] = df

        for update_cell_id, update_data_idx in update_data.items():
            update_data_idx["cell_id"] = update_cell_id
            update_data_idx["cell_lat"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lat"].values[0]
            update_data_idx["cell_lon"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lon"].values[0]
            update_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_az_deg"].values[0]
            update_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            update_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    update_data_idx["cell_az_deg"].values[0],
                    update_data_idx["cell_lat"].values[0],
                    update_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    update_data_idx["latitude"], update_data_idx["longitude"]
                )
            ]
        return update_data

### train_or_update_rf_twin()

In [ ]:
def train_or_update_rf_twin(new_data: pd.DataFrame,topology: pd.DataFrame, bayesian_digital_twins):
        try:
            if not isinstance(new_data, pd.DataFrame):
                raise TypeError("The input 'new_data' must be a pandas DataFrame.")
            
            new_data = new_data.rename(
                columns={"cell_rxpower_dbm": "cell_rxpwr_dbm"}
            )
            expected_columns = {"longitude", "latitude", "cell_rxpwr_dbm"}
            if not expected_columns.issubset(new_data.columns):
                raise ValueError(
                    f"The input DataFrame must contain the following columns: {expected_columns}"
                )

            if bayesian_digital_twins:
                update_data = new_data
                updated_data = _preprocess_ue_update_data(update_data, topology)
                updated_data_list = list(updated_data.values())
                print("Updated_List ",updated_data_list)

                for data_idx, update_data_df in enumerate(updated_data_list):
                    update_cell_id = data_idx + 1
                    if update_cell_id in bayesian_digital_twins:
                        print(f"Updating cell {update_cell_id} with {len(update_data_df)} samples.")
                        bayesian_digital_twins[
                            update_cell_id
                        ].update_trained_gpmodel([update_data_df])
            else:
                print(
                    "No Bayesian Digital Twins available for update. Training from scratch."
                )
                # new_data = new_data.drop(
                #     columns=["cell_rxpower_dbm"], errors="ignore"
                # )
                _training(bayesian_digital_twins,maxiter=100, train_data=new_data,topology = topology)
        
            # return bayesian_digital_twins
        
        except TypeError as te:
            print(f"TypeError: {te}")
        except ValueError as ve:
            print(f"ValueError: {ve}")
        except KeyError as ke:
            print(f"KeyError: {ke}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

## params

In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
params2 = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 7,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 10,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.8,
                    "variance": 0.5,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

## load data

### topology

In [ ]:
topology = pd.read_csv('data/sim_data/topology.csv')

topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90

topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180

topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

### rx power data

[mock_ue_id, latitude, longitude, tick, cell_id, cell_rxpower_dbm]

In [ ]:
combined_df = pd.read_csv('data/sim_data/sorted_rx_data.csv')
combined_df.head()

In [ ]:
update_data = combined_df[['mock_ue_id', 'longitude', 'latitude', 'tick', 'cell_id', 'cell_rxpwr_dbm']]
update_data.head()

## explicit training

In [ ]:
# getting training data from mobility model
# client can use their own data

training_data = get_ue_data(params)
training_data.head()

In [ ]:
bayesian_digital_twins = {}

In [ ]:
bayesian_digital_twins, loss_vs_iters = _training(
    bayesian_digital_twins,
    maxiter=100,
    train_data=training_data,
    topology=topology,
)


In [ ]:
bayesian_digital_twins

### prediction call

to address fantasy model error

In [ ]:
prediction_data = get_ue_data(params2)

In [ ]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bayesian_digital_twins
)

## first update call with empty bdt

In [ ]:
bdt = {}

In [ ]:
updates = train_or_update_rf_twin(update_data,topology,bdt)

## call prediction to address fantasy code error

In [ ]:
prediction_data = get_ue_data(params2)
prediction_data.head()

In [ ]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bdt
)

In [ ]:
predictions.head()

In [ ]:
full_prediction_df.head()

## 2nd update call with trained bdt

In [ ]:
# Training from scratch done now will update the exisitng BDT
update_not_from_scratch = train_or_update_rf_twin(update_data, topology, bdt)

# attempting rx power bypass inside MRO

- preprocess methods are still inside MRO class
- using flags to detect if cell_rxpwr_dbm inside new_data
- validate columns and check if is in combined format
- make bypass changes

## load new_data with rx power

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import pandas as pd
import numpy as np

# from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

In [ ]:
new_data_with_rxpwr = pd.read_csv('data/sim_data/sorted_rx_data.csv')

new_data_with_rxpwr.head()

# this combined data was created from 20_UEs_100_ticks x 3 cells topology data
# so new_data_with_rxpwr is in ue x cell format; contains all of log_distance, relative_bearing, cell_rxpwr_dbm

In [ ]:
new_data_with_rxpwr = new_data_with_rxpwr.loc[:, ['longitude', 'latitude', 'cell_id', 'cell_rxpwr_dbm']]
new_data_with_rxpwr.head()

In [ ]:
new_data_with_rxpwr.shape

## training data

format:

training_data = {cell_1: df1, cell_2: df2, cell_3: df3}

here the 3 dfs are separately imported and then the dict is made.

In [ ]:
cell_1 = pd.read_csv('data/sim_data/training_data_cell_wise/cell_1.csv', index_col=0)

cell_1.head()

In [ ]:
cell_2 = pd.read_csv('data/sim_data/training_data_cell_wise/cell_2.csv', index_col=0)

cell_2.head()

In [ ]:
cell_3 = pd.read_csv('data/sim_data/training_data_cell_wise/cell_3.csv', index_col=0)

cell_3.head()

In [ ]:
training_data = {"cell_1": cell_1, "cell_2": cell_2, "cell_3": cell_3}

# MRO updated flow test for PR fix

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import pandas as pd
from notebooks.radp_library import preprocess_ue_data
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO

## load ue and topology

In [ ]:
simple_ue = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')

In [ ]:
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

simple_ue.drop(columns=['mock_ue_id', 'tick'], inplace=True)

In [ ]:
topology


In [ ]:
simple_ue.head()

## params

In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

## MRO Workflow

### new_data has only lat and lon

In [ ]:
simple_ue.head()

#### calculate rx power data

In [ ]:
# new_data for train_or_update_rf_twin()

input_data = preprocess_ue_data(simple_ue, topology)
input_data.head()

#### initiating MRO

In [ ]:
mro = SimpleMRO(mobility_model_params = params, topology = topology)

In [ ]:
mro.bayesian_digital_twins

#### train bdt from scratch

In [ ]:
mro.train_or_update_rf_twin(input_data)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.solve(n_epochs=10)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
import pickle

# Save the bayesian_digital_twins object to a file
bdt_path = 'data/sim_data/bayesian_digital_twins.pkl'
with open(bdt_path, 'wb') as file:
    pickle.dump(mro.bayesian_digital_twins, file)

print(f"Bayesian Digital Twins saved successfully. at: {bdt_path}")

In [ ]:
mro.topology

In [ ]:
new_data_with_rx_data = preprocess_ue_data(simple_ue, topology)

new_data_with_rx_data = new_data_with_rx_data.loc[:, ['longitude', 'latitude', 'cell_id', 'cell_rxpwr_dbm']]
new_data_with_rx_data.head()

In [ ]:
new_data_with_rx_data.shape

In [ ]:
mro.train_or_update_rf_twin(new_data_with_rx_data)

In [ ]:
def add_cell_info(new_data_with_rx_data, topology):
    """adds cell information ['cell_id', 'cell_lat', 'cell_lon', 'cell_az_deg'] to the DataFrame based on cell_id"""
    new_data_with_rx_data['cell_id'] = new_data_with_rx_data['cell_id'].str.replace('cell_', '').astype(int)
    new_data_with_rx_data_with_cell_info = new_data_with_rx_data.merge(topology[['cell_id', 'cell_lat', 'cell_lon', 'cell_az_deg']], on='cell_id', how='left')
    return new_data_with_rx_data_with_cell_info

In [ ]:
cartesian_rx_data = add_cell_info(new_data_with_rx_data, topology)
cartesian_rx_data.head()

In [ ]:
cartesian_rx_data.shape

In [ ]:
from notebooks.radp_library import calc_log_distance, calc_relative_bearing

In [ ]:
calc_log_distance(cartesian_rx_data)

In [ ]:
calc_relative_bearing(cartesian_rx_data)